In [14]:
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import open3d as o3d
from plicalib import meshes
from plicalib.folds import FoldAnnotation
from sklearn.decomposition import PCA

In [2]:
mesh = o3d.io.read_triangle_mesh("examples/legdisc/20250207_ecadGFP_legdisc_2hAPF_disc1.ply")
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
vertex_normals = meshes.compute_vertex_normals(vertices, triangles)
triangle_normals = meshes.compute_face_normals(vertices, triangles)
triangle_centers = vertices[triangles].mean(axis=1)

In [9]:
def meshes_squared_distance(tc_1, n_1, tc_2, n_2, sigma):
    cross_c_sd = (tc_1**2).sum(dim=-1)[:, None] + (tc_2**2).sum(dim=-1)[None, :] - 2 * tc_1 @ tc_2.T
    cross_weights = torch.exp(-cross_c_sd / (2 * sigma**2))
    cross_term = torch.sum(cross_weights * (n_1 @ n_2.T)**2)
    self_c_sd_1 = (tc_1**2).sum(dim=-1)[:, None] + (tc_1**2).sum(dim=-1)[None, :] - 2 * tc_1 @ tc_1.T
    self_weights_1 = torch.exp(-self_c_sd_1 / (2 * sigma**2)) 
    self_term_1 = torch.sum(self_weights_1 * (n_1 @ n_1.T)**2)
    self_c_sd_2 = (tc_2**2).sum(dim=-1)[:, None] + (tc_2**2).sum(dim=-1)[None, :] - 2 * tc_2 @ tc_2.T
    self_weights_2 = torch.exp(-self_c_sd_2 / (2 * sigma**2)) 
    self_term_2 = torch.sum(self_weights_2 * (n_2 @ n_2.T)**2)
    return self_term_1 + self_term_2 - 2 * cross_term

def meshes_inner_product_chuncked(tc_1, n_1, tc_2, n_2, sigma, chunck_size):
    total = 0.0
    for start in range(0, tc_1.shape[0], chunck_size):
        end = min(start + chunck_size, tc_1.shape[0])
        tc_2_chunk = tc_2[start:end]
        n_2_chunk = n_2[start:end]
        sd = (tc_1**2).sum(dim=-1)[:, None] + (tc_2_chunk**2).sum(dim=-1)[None, :] - 2 * tc_1 @ tc_2_chunk.T
        weights = torch.exp(-sd / (2 * sigma**2))
        total += torch.sum(weights * (n_1 @ n_2_chunk.T)**2)
    return total

In [20]:
mesh_pca = PCA(n_components=3).fit(vertices)
mesh_typical_size = np.sqrt(mesh_pca.explained_variance_.sum())

In [21]:
sphere_mesh = o3d.geometry.TriangleMesh.create_sphere(radius=mesh_typical_size, resolution=20)
sphere_mesh.compute_vertex_normals()
vertices_sphere = np.asarray(sphere_mesh.vertices)
triangles_sphere = np.asarray(sphere_mesh.triangles)
triangle_normals_sphere = meshes.compute_face_normals(vertices_sphere, triangles_sphere)
triangle_centers_sphere = vertices_sphere[triangles_sphere].mean(axis=1)

In [ ]:
num_iterations = 10
num_control_points = 10
sigma = mesh_typical_size * 0.1
alpha = torch.zeros(num_control_points, 3, requires_grad=True)
control_points = 
target_centers = torch.tensor(triangle_centers, dtype=torch.float32, requires_grad=False)
target_normals = torch.tensor(triangle_normals, dtype=torch.float32, requires_grad=False)
source_centers = torch.tensor(triangle_centers_sphere, dtype=torch.float32, requires_grad=True)
source_normals = torch.tensor(triangle_normals_sphere, dtype=torch.float32, requires_grad=False)
for iteration in range(num_iterations):
    squared_distance = meshes_inner_product_chuncked(source_centers, source_normals, source_centers, source_normals, sigma=sigma, chunck_size=100) 
    squared_distance += meshes_inner_product_chuncked(target_centers, target_normals, target_centers, target_normals, sigma=sigma, chunck_size=100)
    squared_distance -= 2 * meshes_inner_product_chuncked(source_centers, source_normals, target_centers, target_normals, sigma=sigma, chunck_size=100)
    break